# Day 1 Lab: Responsible Data Controls in Finance
Welcome to the first hands-on block of the mini-MBA. Run these cells in Google Colab (Runtime → Run all) and replace placeholder paths with your Bloomberg CSV exports.

> **Learning outcomes**
> - mount data securely (Drive or manual upload)
> - build a lightweight data dictionary & lineage tracker
> - run automated quality checks on ESG disclosure data
> - summarize findings for a compliance-ready log

## 0. Colab runtime prep
1. Switch runtime to **Python 3** (default).
2. (Optional) Enable GPU is not needed today; CPU tier is sufficient.
3. Decide how you will provide Bloomberg data:
   - **Google Drive**: upload CSV/XLSX exports to a folder, then mount Drive.
   - **Manual upload**: use `files.upload()` and store them under `/content/data`.
4. Create a subfolder per dataset using the naming convention `datasetname_YYYYMMDD.csv`.

In [ ]:
%%capture
!pip install pandas pyjanitor plotly==5.24.0 rich openpyxl

In [ ]:
from pathlib import Path
from datetime import datetime

try:
    from google.colab import drive, files  # type: ignore
    IN_COLAB = True
except ImportError:  # local dev fallback
    drive = None
    files = None
    IN_COLAB = False

DATA_ROOT = Path("/content/data") if IN_COLAB else Path.cwd() / "data" / "bloomberg"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_LOG = Path("/content/data_access_log.csv") if IN_COLAB else Path.cwd() / "data" / "data_access_log.csv"

if IN_COLAB:
    print("Colab runtime detected.")
    if not (Path("/content/drive").exists() and any(Path("/content/drive").iterdir())):
        print("Mounting Google Drive at /content/drive")
        drive.mount("/content/drive", force_remount=True)
else:
    print(f"Running outside Colab. DATA_ROOT={DATA_ROOT}")

### Upload helper
Run one of the helpers below to get Bloomberg exports into `/content/data`. Feel free to skip if you already have the files locally.

In [ ]:
def copy_from_drive(src_folder: str, pattern: str = "*.csv"):
    """Copy Bloomberg exports from Drive into the working directory."""
    if not IN_COLAB:
        raise RuntimeError("Drive helper only works inside Colab")
    import shutil
    from glob import glob

    src = Path("/content/drive/MyDrive") / src_folder
    matches = list(src.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No files matching {pattern} in {src}")
    for path in matches:
        dest = DATA_ROOT / path.name
        shutil.copy(path, dest)
        print(f"Copied {path.name} -> {dest}")

In [ ]:
def manual_upload():
    """Fallback uploader if Drive mounting is not possible."""
    if not IN_COLAB:
        raise RuntimeError("Use regular file copy when running locally.")
    uploaded = files.upload()
    for filename, data in uploaded.items():
        dest = DATA_ROOT / filename
        with dest.open("wb") as fh:
            fh.write(data)
        print(f"Saved {filename} -> {dest}")

## 1. Describe your Bloomberg ESG dataset
- Recommended source: `ESG <GO>` → **Fundamentals** tab.
- Suggested fields: `TICKER`, `COUNTRY`, `INDUSTRY_SECTOR`, `ENV_DISCLOSURE_SCORE`, `SOC_DISCLOSURE_SCORE`, `GOV_DISCLOSURE_SCORE`, `LAST_UPDATE_DT`.
- Save your export as `esg_scores_YYYYMMDD.csv`.

Use the cells below to build a lightweight data dictionary and lineage record.

In [ ]:
import pandas as pd
import json

ESG_PATH = DATA_ROOT / "esg_scores_SAMPLE.csv"  # TODO: rename to your file

schema = {
    "dataset_name": ESG_PATH.stem,
    "source": "Bloomberg ESG <GO> export",
    "extraction_timestamp": datetime.utcnow().isoformat(),
    "fields": [
        {"name": "ticker", "description": "Bloomberg Ticker", "pii": False},
        {"name": "env_disclosure_score", "description": "Environmental disclosure percentile", "pii": False},
        {"name": "soc_disclosure_score", "description": "Social disclosure percentile", "pii": False},
        {"name": "gov_disclosure_score", "description": "Governance disclosure percentile", "pii": False},
        {"name": "last_update_dt", "description": "Bloomberg reported update date", "pii": False},
    ]
}
print(json.dumps(schema, indent=2))

In [ ]:
df = pd.read_csv(ESG_PATH)
df.head()

## 2. Automated quality checks
Build a reusable suite that flags:
- schema mismatches
- missing values & zeros
- out-of-range disclosure scores (should be 0–100)
- duplicate tickers or stale timestamps

In [ ]:
from rich.console import Console
from rich.table import Table

console = Console()

EXPECTED_COLUMNS = {
    "ticker", "env_disclosure_score", "soc_disclosure_score",
    "gov_disclosure_score", "industry_sector", "country", "last_update_dt"
}


def quality_report(frame: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    checks = []
    columns = set(map(str.lower, frame.columns))

    missing_cols = EXPECTED_COLUMNS - columns
    if missing_cols:
        checks.append({"check": "schema_columns", "status": "fail", "detail": f"Missing: {sorted(missing_cols)}"})
    else:
        checks.append({"check": "schema_columns", "status": "pass", "detail": "All expected columns present"})

    for col in [c for c in frame.columns if c.lower().endswith("_score")]:
        out = frame[~frame[col].between(0, 100)]
        status = "fail" if not out.empty else "pass"
        detail = f"{len(out)} rows out of range" if not out.empty else "0 rows out of range"
        checks.append({"check": f"range_{col}", "status": status, "detail": detail})

    dupes = frame.duplicated(subset=["ticker"]).sum()
    checks.append({"check": "duplicate_ticker", "status": "fail" if dupes else "pass", "detail": f"{dupes} duplicates"})

    stale = frame[frame["last_update_dt"] < "2022-01-01"].shape[0]
    checks.append({"check": "stale_updates", "status": "fail" if stale else "pass", "detail": f"{stale} rows before 2022"})

    report = pd.DataFrame(checks)
    table = Table(title=f"Quality report: {dataset_name}")
    table.add_column("Check")
    table.add_column("Status")
    table.add_column("Detail")
    for _, row in report.iterrows():
        table.add_row(row.check, row.status, row.detail)
    console.print(table)
    return report

In [ ]:
quality_df = quality_report(df, dataset_name=schema["dataset_name"])
quality_df

## 3. Compliance log
This block appends an auditable entry capturing who accessed the data, when, and what validation status they observed.

In [ ]:
log_entry = {
    "timestamp": datetime.utcnow().isoformat(),
    "user": "YOUR_NAME",  # edit!
    "dataset": schema["dataset_name"],
    "file_path": str(ESG_PATH),
    "quality_status": "fail" if (quality_df["status"] == "fail").any() else "pass",
    "notes": "Initial ingestion"
}
log_df = pd.DataFrame([log_entry])
if DATA_LOG.exists():
    existing = pd.read_csv(DATA_LOG)
    log_df = pd.concat([existing, log_df], ignore_index=True)
log_df.to_csv(DATA_LOG, index=False)
log_df.tail(5)

### Deliverable
Export the notebook as PDF or HTML from Colab, attach the `data_access_log.csv`, and write a three-bullet summary answering:
1. Which checks failed?
2. What remediation is needed (re-download, contact data desk, transform)?
3. How will you store the dataset going forward?